In [1]:
# =====================================================================
# CÉLULA 1 — Importações
# =====================================================================
import os
import re
import sys
import unicodedata
from datetime import date, datetime, time, timedelta
from pathlib import Path

import pandas as pd

ENCODINGS_TENTATIVA = ("utf-8-sig", "utf-8", "latin-1", "cp1252")
SEPARADORES_TENTATIVA = (";", ",", "\t", "|")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)

In [2]:
# =====================================================================
# CÉLULA 2 — Caminhos dinâmicos
# =====================================================================
def get_base_dir() -> Path:
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent

    vsc = globals().get("__vsc_ipynb_file__")
    if vsc:
        return Path(vsc).resolve().parent

    arquivo = globals().get("__file__")
    if arquivo:
        return Path(arquivo).resolve().parent

    return Path.cwd()


BASE_DIR = get_base_dir()


def resolver_pasta_arquivos(nome: str = "arquivos") -> Path:
    """1. AFD_PASTA  2. <base>/arquivos  3. <_MEIPASS>/arquivos  4. cria"""
    env = os.getenv("AFD_PASTA")
    if env:
        return Path(env).expanduser().resolve()

    candidatos = [BASE_DIR / nome]
    meipass = getattr(sys, "_MEIPASS", None)
    if meipass:
        candidatos.append(Path(meipass) / nome)

    for candidato in candidatos:
        if candidato.is_dir():
            return candidato

    destino = BASE_DIR / nome
    destino.mkdir(parents=True, exist_ok=True)
    return destino


PASTA_ARQUIVOS = resolver_pasta_arquivos()

print("BASE_DIR       :", BASE_DIR)
print("PASTA_ARQUIVOS :", PASTA_ARQUIVOS)

BASE_DIR       : \\AGROSRV\Automacoes\Marcacao_Pontos
PASTA_ARQUIVOS : \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos


In [3]:
# =====================================================================
# CÉLULA 3 — Arquivos CSV encontrados
# =====================================================================
EXTENSOES_CSV = {".csv", ".txt"}


def parece_relatorio_rhid(caminho: Path) -> bool:
    """Confere se a 1ª linha tem cara de cabeçalho do relatório do RHiD."""
    for enc in ENCODINGS_TENTATIVA:
        try:
            with caminho.open("r", encoding=enc, errors="strict") as f:
                cabecalho = f.readline().lower()
            break
        except UnicodeDecodeError:
            continue
    else:
        return False

    return ("cpf" in cabecalho or "funcionario" in cabecalho) and "dia" in cabecalho


def listar_relatorios(pasta: Path | str | None = None) -> list[Path]:
    pasta = Path(pasta or PASTA_ARQUIVOS)
    if not pasta.is_dir():
        return []
    return [
        item for item in sorted(pasta.iterdir())
        if item.is_file()
        and item.suffix.lower() in EXTENSOES_CSV
        and parece_relatorio_rhid(item)
    ]


ARQUIVOS = listar_relatorios()
print(f"{len(ARQUIVOS)} arquivo(s) em {PASTA_ARQUIVOS}")
for arq in ARQUIVOS:
    print(f"  - {arq.name}  ({arq.stat().st_size / 1024:.1f} KB)")

1 arquivo(s) em \\AGROSRV\Automacoes\Marcacao_Pontos\arquivos
  - 2026 - Outubro.CSV  (58.4 KB)


In [4]:
# =====================================================================
# CÉLULA 4 — Leiaute, colunas e parâmetros legais
# =====================================================================
CAMPOS_FIXOS = {
    "nome": ("nome_do_funcionario", "nome_funcionario", "funcionario", "nome", "colaborador"),
    "cpf": ("cpf_do_funcionario", "cpf_funcionario", "cpf", "cpf_colaborador"),
    "dia": ("dia", "data", "data_dia"),
    "previsto": ("previsto", "horario_previsto", "jornada_prevista", "escala", "horario_contratual"),
    "total_normais": ("total_normais", "normais", "total_normal"),
    "total_noturno": ("total_noturno", "noturno"),
    "dia_falta": ("dia_falta", "falta_dia"),
    "falta_e_atraso": ("falta_e_atraso", "faltas_e_atrasos", "falta_atraso"),
    "abono": ("abono", "abonos"),
    "extra_0_d": ("extra_0_d", "extra_0d"),
    "extra_60_d": ("extra_60_d", "extra_60d"),
    "extra_diurna": ("extra_diurna", "extras_diurna"),
    "extra_noturna": ("extra_noturna", "extras_noturna"),
    "banco_total": ("banco_total", "total_banco", "banco_dia"),
    "banco_saldo": ("banco_saldo", "saldo_banco", "saldo"),
    "justificativas": ("justificativas", "justificativa", "observacoes", "obs"),
}

CAMPOS_DURACAO = (
    "total_normais", "total_noturno", "dia_falta", "falta_e_atraso", "abono",
    "extra_0_d", "extra_60_d", "extra_diurna", "extra_noturna",
    "banco_total", "banco_saldo",
)

DIAS_ABREV = ("seg", "ter", "qua", "qui", "sex", "sáb", "dom")

# Intervalo NÃO remunerado após cada par de marcação:
#   após o par 1 (Saída 1 -> Entrada 2) = Café
#   após o par 2 (Saída 2 -> Entrada 3) = Almoço
ROTULO_INTERVALO = {1: "Café", 2: "Almoço", 3: "Intervalo 3", 4: "Intervalo 4"}

# Qual intervalo é o intrajornada legal (art. 71). No RHiD é o 2.
INDICE_INTERVALO_ALMOCO = 2

# ---------------------------------------------------------------------
# PARÂMETROS LEGAIS — revise contra a CCT da categoria antes de usar
# ---------------------------------------------------------------------
PARAMETROS_LEGAIS = {
    # CLT art. 58, §1º
    "tolerancia_min_por_marcacao": 5,
    "tolerancia_min_por_dia": 10,

    # CLT art. 71, caput
    "jornada_longa": timedelta(hours=6),
    "jornada_media": timedelta(hours=4),
    "intervalo_legal_longo": timedelta(hours=1),
    "intervalo_legal_medio": timedelta(minutes=15),

    # CLT art. 611-A, III + Tema 1046 STF (CCT pode reduzir até 30 min)
    "usar_minimo_cct": False,
    "intervalo_minimo_cct": timedelta(minutes=30),

    # CLT art. 73, §§1º e 2º + Súmula 60 TST
    "noturno_inicio": time(22, 0),
    "noturno_fim": time(5, 0),
    "fator_hora_noturna": 52.5,
    "adicional_noturno": 0.20,
    "aplicar_sumula_60": True,

    # CLT art. 59, caput
    "limite_extra_dia": timedelta(hours=2),

    # CLT art. 71, §4º
    "percentual_intervalo_suprimido": 0.50,

    # Alerta de marcação muito distante do previsto
    "alerta_desvio_min": 60,
}


def normalizar_nome_coluna(nome) -> str:
    texto_ = str(nome or "").strip().lower()
    texto_ = unicodedata.normalize("NFKD", texto_)
    texto_ = "".join(c for c in texto_ if not unicodedata.combining(c))
    texto_ = re.sub(r"[^a-z0-9]+", "_", texto_)
    return texto_.strip("_")


def resolver_colunas(colunas) -> dict[str, str]:
    mapa_normalizado = {normalizar_nome_coluna(c): c for c in colunas}
    resolvido = {}

    for campo, candidatos in CAMPOS_FIXOS.items():
        for candidato in candidatos:
            if candidato in mapa_normalizado:
                resolvido[campo] = mapa_normalizado[candidato]
                break
        else:
            for candidato in candidatos:
                achado = next(
                    (real for norm, real in mapa_normalizado.items()
                     if norm.startswith(candidato)), None)
                if achado:
                    resolvido[campo] = achado
                    break

    return resolvido


def mapear_pares(colunas) -> dict[int, dict[str, str]]:
    pares: dict[int, dict[str, str]] = {}

    for coluna in colunas:
        norm = normalizar_nome_coluna(coluna)

        achado = re.match(r"^(entrada|ent)_?(\d+)$", norm)
        if achado:
            pares.setdefault(int(achado.group(2)), {})["entrada"] = coluna
            continue

        achado = re.match(r"^(saida|sai)_?(\d+)$", norm)
        if achado:
            pares.setdefault(int(achado.group(2)), {})["saida"] = coluna

    return dict(sorted(pares.items()))


def conferir_leiaute(df: pd.DataFrame) -> tuple[dict, dict, list[str]]:
    campos = resolver_colunas(df.columns)
    pares = mapear_pares(df.columns)

    problemas = []
    for obrigatorio in ("nome", "cpf", "dia"):
        if obrigatorio not in campos:
            problemas.append(
                f"coluna obrigatória não encontrada: {obrigatorio}")

    if not pares:
        problemas.append("nenhum par entrada/saída encontrado")

    return campos, pares, problemas